### Fetching Live Data from a Web API (requests)

`requests` is a library that lets Python "visit" a website or API,
just like a browser does, and get data back.
### Making the request

### Reading the data

### Looping through the results

### Key takeaway
- requests.get(url)  -> fetches data from a web API
- response.status_code -> tells you if it worked (200 = success)
- response.json()      -> converts the reply into usable Python data
- Once it's a list of dictionaries, all your normal Python skills
  (loops, f-strings, indexing) work exactly the same as always.

In [1]:
### Making the request

import requests

url = "https://api.frankfurter.dev/v2/rates?quotes=USD,PKR"

response = requests.get(url)                  # sends the request, gets a response back
print(response)                 

<Response [200]>


In [2]:
dir(response)

['__annotations__',
 '__attrs__',
 '__bool__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__nonzero__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_content',
 '_content_consumed',
 '_next',
 'apparent_encoding',
 'close',
 'connection',
 'content',
 'cookies',
 'elapsed',
 'encoding',
 'headers',
 'history',
 'is_permanent_redirect',
 'is_redirect',
 'iter_content',
 'iter_lines',
 'json',
 'links',
 'next',
 'ok',
 'raise_for_status',
 'raw',
 'reason',
 'request',
 'status_code',
 'text',
 'url']

In [3]:
print(response.status_code)  #---> response.status_code — 200 means success. Other codes (404, 500, etc.) mean something went wrong.              

200


In [4]:
print(response.json())
#---> response.json() — converts the raw response text into a Python list/dict

[{'date': '2026-09-10', 'base': 'EUR', 'quote': 'PKR', 'rate': 323.47}, {'date': '2026-09-10', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1645}]


In [5]:
def get_exchange_rate(base, target):
    try:
        url = f"https://api.frankfurter.dev/v2/rate/{base}/{target}"
        response = requests.get(url)
        print(response.status_code)
        data = response.json()
        print(data)
        return data["rate"]
    except Exception as e:
        print(f"Something went wrong: {e}")
        return None

result = get_exchange_rate("EUR", "PKR")
print(result)
result = get_exchange_rate("EUR", "FAKE")
print(result)

200
{'date': '2026-09-10', 'base': 'EUR', 'quote': 'PKR', 'rate': 323.47}
323.47
422
{'status': 422, 'message': 'invalid currency: FAKE'}
Something went wrong: 'rate'
None


In [6]:
print(get_exchange_rate("USD", "GBP"))
print(get_exchange_rate("PKR", "EUR"))

200
{'date': '2026-09-10', 'base': 'USD', 'quote': 'GBP', 'rate': 0.73704}
0.73704
200
{'date': '2026-09-10', 'base': 'PKR', 'quote': 'EUR', 'rate': 0.00309}
0.00309


## Currency Converter + AI Travel Tip Generator

Fetches a live exchange rate via the Frankfurter API, converts an amount, 
and asks Gemini to generate a one-line travel tip using the result — 
combining a live API call with an LLM prompt.

In [11]:
def currency_helper(base, target, amount):
    rate = get_exchange_rate(base, target)
    
    if rate is None:
        print("Couldn't get the rate.")
        return
    
    converted = amount * rate
    print(f"{amount} {base} = {converted} {target}")
    
    prompt = f"Write a one-line travel tip that mentions spending {converted} {target}."
    
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    
    print(response.text)


currency_helper("EUR", "PKR", 100)

200
{'date': '2026-09-10', 'base': 'EUR', 'quote': 'PKR', 'rate': 323.43}
100 EUR = 32343.0 PKR
To elevate your trip to Northern Pakistan, spend 32343.0 PKR to secure a premium cabin stay in Hunza Valley for unmatched views of the majestic Rakaposhi peak.
